# LAB 4b:  Keras による DNN モデルの作成

**学習目標**

1. CSV 列、ラベル列、列のデフォルト値の設定
1. CSV ファイルから特徴量とラベルのデータセットを作成
1. 生の特徴量に対する入力レイヤを作成
1. DNN の全結合（Dense）隠れ層と出力レイヤを作成
1. カスタム評価指標を作成
1. すべての構成要素を結合して DNN モデルを構築
1. モデルのトレーニングと評価


## はじめに 
このノートブックでは、Keras を使用して、赤ちゃんが生まれる前の体重を予測する DNN モデルを作成します。

まず、データの入力用に CSV 列名、ラベル列、および列のデフォルト値を定義します。次に、CSV ファイルから特徴量とラベルの tf.data Dataset を構築し、生の特徴量に対する入力レイヤを作成します。続いて、モデル入力用の前処理レイヤを設定し、Keras でディープ ニューラル ネットワークを構築します。カスタム評価指標を作成して DNN モデルを構築します。最後に、モデルのトレーニングと評価を行います。

各学習目標は、この演習用ラボ ノートブック内の __#TODO__ に対応しています。まずはこのノートブックを自身で完成させてみてから、[解説ノートブック（solution）](../solutions/4b_keras_dnn_babyweight.ipynb) を確認してください。

## 必要なライブラリの読み込み

In [ ]:
import datetime
import os
import shutil

import keras
import matplotlib.pyplot as plt
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

## CSV ファイルが存在することの確認

このシリーズの 7 番目のラボ [4a_sample_babyweight](../solutions/4a_sample_babyweight.ipynb) では、BigQuery からトレーニング用、評価用、テスト用の CSV ファイルをサンプリングしました。それらのファイルが存在することを確認してください。存在しない場合は、そのラボに戻って作成してください。

In [ ]:
%%bash
ls *.csv

In [ ]:
%%bash
head -5 *.csv

## Keras モデルの作成

### ラボ課題 1: CSV 列、ラベル列、列のデフォルト値の設定

CSV ファイルが存在することを確認できたので、入力関数で使用するいくつかの設定を行います。
* `CSV_COLUMNS`: 列のヘッダー名。CSV ファイルと同じ順序になっていることを確認してください。
* `LABEL_COLUMN`: ラベルとなる列のヘッダー名。特徴量ディクショナリからポップ（取り出し）するために必要です。
* `DEFAULTS`: `CSV_COLUMNS` と同じ長さのリスト。すなわち、CSV の各列に対するデフォルト値です。各要素自体が、その CSV 列のデフォルト値を含むリストになります。

In [ ]:
# CSV、ラベル、キーの各列を決定
# TODO: 文字列の列ヘッダーのリストを作成し、順序が一致していることを確認する
CSV_COLUMNS = [""]

# ラベル列の文字列名を追加
# TODO: ラベル列の文字列名を追加する
LABEL_COLUMN = ""

NUMERICAL_COLUMNS = ["mother_age", "gestation_weeks"]
CATEGORICAL_COLUMNS = ["is_male", "plurality"]

# 各 CSV 列のデフォルト値をリストのリストとして設定
# is_male と plurality は文字列として扱う

# TODO: デフォルト値を定義する
DEFAULTS = []

### ラボ課題 2: CSV ファイルから特徴量とラベルのデータセットを作成

### CSV ファイルから特徴量とラベルのデータセットを作成

次に、データを読み込むための `input_fn` を作成します。これにより CSV データセット オブジェクトが作成されます。ただし、列を特徴量とラベルに分割する必要があります。これを行うには、データセットに map メソッドを適用し、特徴量テンソルのディクショナリからラベル列を pop します。

In [ ]:
def features_and_labels(row_data):
    """特徴量ディクショナリから特徴量とラベルを分割します。

    Args:
        row_data: CSV 列名とテンソル値のディクショナリ。
    Returns:
        特徴量テンソルのディクショナリとラベル テンソル。
    """
    label = row_data.pop(LABEL_COLUMN)

    return row_data, label  # 特徴量、ラベル


def decode_csv(line):
    fields = tf.io.decode_csv(line, record_defaults=DEFAULTS)
    return dict(zip(CSV_COLUMNS, fields))


def load_dataset(pattern, batch_size=1, mode="eval"):
    """CSV ファイルから tf.data API を使用してデータセットを読み込みます。

    Args:
        pattern: str、ファイル一覧に glob 展開するファイル パターン。
        batch_size: int、バッチあたりのサンプル数。
        mode: トレーニングか評価かを指定。
    Returns:
        `Dataset` オブジェクト。
    """
    # これらのファイルの内容を読み込むために TextLineDataset を使用
    # また、複数のファイルから並列に読み込むために interleave を使用
    dataset = None

    # 各 CSV ファイルを読み込むために decode_csv を呼び出す
    dataset = None
    # 各 CSV ファイルをマッピングするために features_and_labels を呼び出す
    dataset = None
    dataset = dataset.batch(batch_size)
    # トレーニング用にシャッフル
    if mode == "train":
        dataset = dataset.shuffle(buffer_size=1000)

    dataset = dataset.repeat().prefetch(buffer_size=tf.data.AUTOTUNE)

    return dataset

### ラボ課題 3: 生の特徴量に対する入力レイヤの作成

入力関数によって読み込まれたデータをモデル関数に渡す必要がありますが、どのように連携させればよいでしょうか。以下を定義して Keras の入力レイヤを使用できます。
* shape: バッチサイズを含まない shape タプル（整数値）。たとえば、shape=(32,) は期待される入力が 32 次元のベクトルのバッチであることを示します。このタプルの要素には None を指定できます。'None' の要素は shape が不明な次元を表します。
* name: レイヤの任意の名前文字列。モデル内で一意である必要があります（同じ名前を 2 回再利用しないでください）。指定しない場合は自動生成されます。
* dtype: 入力に期待されるデータ型を表す文字列（float32, float64, int32 など）。

In [ ]:
def create_input_layers():
    """各特徴量の入力レイヤのディクショナリを作成します。

    Returns:
        各特徴量の `Keras.layers.Input` レイヤのディクショナリ。
    """
    # TODO: 数値列（NUMERIC COLUMNS）用の keras.layers.Input のディクショナリを作成する
    inputs = {}

    # TODO: カテゴリ列（CATEGORICAL COLUMNS）用の keras.layers.Input のディクショナリを追加する
    inputs.update()

    return inputs

### ラボ課題 4: 入力用の前処理レイヤの定義

次に、生の特徴量を変換するための前処理レイヤを定義します。`mother_age` と `gestation_weeks` は数値型である必要があります。その他の `is_male` と `plurality` はカテゴリ型である必要があります。

In [ ]:
def transform(inputs):
    """変換された入力のディクショナリを作成します。

    Returns:
        変換された Tensor のディクショナリ
    """

    transformed = {}

    for numerical_column in NUMERICAL_COLUMNS:
        transformed[numerical_column] = inputs[numerical_column]

    vocab = {
        "is_male": ["True", "False", "Unknown"],
        "plurality": [
            "Single(1)",
            "Twins(2)",
            "Triplets(3)",
            "Quadruplets(4)",
            "Quintuplets(5)",
            "Multiple(2+)",
        ],
    }

    # TODO: カテゴリ列をワンホット エンコードするために StringLookup レイヤを追加する
    for categorical_column in CATEGORICAL_COLUMNS:
        transformed[categorical_column] = # TODO

    return transformed

### ラボ課題 5: DNN の全結合（Dense）隠れ層と出力レイヤの作成

機械学習用に入力を準備する方法を理解できたので、次はそれらを期待する出力に接続する必要があります。モデル アーキテクチャがその 2 つを結び付けます。入力から始まるいくつかの隠れ全結合層を作成し、全結合の出力レイヤで終了しましょう。これは回帰であるため、出力レイヤの活性化関数が適切であること、および shape が正しいことを確認してください。

In [ ]:
def get_model_outputs(inputs):
    """モデル アーキテクチャを作成し、出力を返します。

    Args:
        inputs: モデルへの入力として使用される Dense テンソル。
    Returns:
        モデルからの Dense テンソル出力。
    """
    # TODO: [64, 32] の 2 つの隠れ層を作成する

    # TODO: 最終出力レイヤを作成する
    output = None

    return output

### ラボ課題 6: すべての構成要素を結合して DNN モデルを構築

素晴らしい！すべてのパーツが揃ったので、あとはそれらをすべてまとめて Keras Model に結合するだけです。これは分岐やサイド入力などのないシンプルなフィードフォワード モデルであるため、Keras の Sequential Model API を使用することもできましたが、練習として Keras の Functional Model API を使用します。ここでは `keras.models.Model` を使用して入力と出力を指定してモデルを構築し、オプティマイザ、損失関数、評価指標を指定してモデルをコンパイルします。

In [ ]:
def build_dnn_model():
    """Keras Functional API を使用してシンプルな DNN を構築します。

    Returns:
        `keras.models.Model` オブジェクト。
    """
    # 入力レイヤを作成
    inputs = create_input_layers()

    # transform
    transformed = transform(inputs)

    # Keras の Functional API では次が必要: LayerConstructor()(inputs)
    dnn_inputs = keras.layers.Concatenate()(list(transformed.values()))

    # 入力を考慮したモデルの出力を取得
    output = get_model_outputs(dnn_inputs)

    # モデルを構築し、すべてをまとめてコンパイル
    # TODO: リストに rmse と mse の評価指標を追加する
    model = keras.models.Model(inputs=inputs, outputs=output)
    rmse = keras.metrics.RootMeanSquaredError(name)
    model.compile(optimizer="adam", loss="mse", metrics=[])

    return model


print("Here is our DNN architecture so far:\n")
model = build_dnn_model()
print(model.summary())

Keras の plot_model ユーティリティを使用して DNN を可視化できます。

In [ ]:
keras.utils.plot_model(model=model, show_layer_names=True, rankdir="LR")

## モデルの実行と評価

### ラボ課題 7: トレーニングと評価

CSV ファイルからの入力と設計したアーキテクチャを使用して Keras モデルを構築しました。ここで、モデル パラメータをトレーニングし、定期的に評価を実行して、トレーニングが進むにつれて未知の外部データに対してどの程度うまく機能しているかを追跡することで、モデルを実行しましょう。train データセットと eval データセットの両方を読み込み、fit メソッドを通じてそれらをモデルに渡す必要があります。データを読み込むときは、適切なパターン、バッチサイズ、モードを指定していることを確認してください。また、TensorBoard へのコールバックを追加することも忘れないでください。

In [ ]:
TRAIN_BATCH_SIZE = 32
NUM_TRAIN_EXAMPLES = 10000 * 5  # トレーニング データセットはリピートされ、周回します
NUM_EVALS = 5  # 評価を行う回数
# 妥当なサンプルを得るのに十分で、かつ処理が遅くならない程度の量
NUM_EVAL_EXAMPLES = 10000

trainds = load_dataset(
    pattern="train*",
    batch_size=TRAIN_BATCH_SIZE,
    mode="train",
)

evalds = load_dataset(pattern="eval*", batch_size=1000, mode="eval").take(
    count=NUM_EVAL_EXAMPLES // 1000
)

steps_per_epoch = NUM_TRAIN_EXAMPLES // (TRAIN_BATCH_SIZE * NUM_EVALS)

logdir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=logdir, histogram_freq=1
)

# TODO: トレーニング データセットでモデルを適合（fit）させ、定期的に評価する
history = model.fit()

### 損失曲線の可視化

In [ ]:
# プロット
import matplotlib.pyplot as plt

nrows = 1
ncols = 2
fig = plt.figure(figsize=(10, 5))

for idx, key in enumerate(["loss", "root_mean_squared_error"]):
    ax = fig.add_subplot(nrows, ncols, idx + 1)
    plt.plot(history.history[key])
    plt.plot(history.history[f"val_{key}"])
    plt.title(f"model {key}")
    plt.ylabel(key)
    plt.xlabel("epoch")
    plt.legend(["train", "validation"], loc="upper left")

### モデルの保存

In [ ]:
OUTPUT_DIR = "babyweight_trained"
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

EXPORT_PATH = os.path.join(OUTPUT_DIR, TIMESTAMP)

SAVEDMODEL_PATH = os.path.join(EXPORT_PATH, "savedmodel")
KERAS_PATH = os.path.join(EXPORT_PATH, "model.keras")

model.export(EXPORT_PATH)
model.save(KERAS_PATH)

In [ ]:
!ls $EXPORT_PATH

## ラボのまとめ: 
このラボでは、まずデータの入力用に CSV 列名、ラベル列、および列のデフォルト値を定義しました。次に、CSV ファイルから特徴量とラベルの tf.data Dataset を構築し、生の特徴量に対する入力レイヤを作成しました。続いて、モデル入力用の前処理レイヤを設定し、Keras でディープ ニューラル ネットワークを構築しました。カスタム評価指標を作成して DNN モデルを構築しました。最後に、モデルのトレーニングと評価を行いました。

Copyright 2026 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.